# Homework — Math Score vs Parental level of education

Dataset: [Students Performance in Exams](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams)

Goal: the dataset has many columns, but per the assignment we focus on just two: predict `math score` from
`parental level of education` with a simple linear regression, and report how accurate that is.

Task checklist: **A)** clean data · **B)** build model · **C)** measure accuracy · **D)** document (this notebook).

## A. Load & Clean the Data

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.precision", 2)

In [2]:
raw = pd.read_csv("../data/students_performance.csv")
raw.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
# Check for missing values / duplicate rows on the FULL row first -- checking only our two
# columns of interest would flag unrelated students who just happen to share a gender + score
# as "duplicates", which is wrong.
print("shape:", raw.shape)
print("missing values:\n", raw.isna().sum().sum(), "total")
print("duplicate rows (full record):", raw.duplicated().sum())

shape: (1000, 8)
missing values:
 0 total
duplicate rows (full record): 0


In [4]:
df = raw.drop_duplicates()[["parental level of education", "math score"]].copy()
df["edu_code"] = df["parental level of education"].map({"some high school": 0, "high school": 1, "some college": 2, "associate's degree": 3, "bachelor's degree": 4, "master's degree": 5})

print("edu values:", df["edu_code"].unique())
df.describe(include="all")

edu values: [4 2 5 3 1 0]


,parental level of education,math score,edu_code
count,1000,1000.00,1000.00
unique,6,NaN,NaN
top,some college,NaN,NaN
freq,226,NaN,NaN
mean,NaN,66.09,2.08
std,NaN,15.16,1.46
min,NaN,0.00,0.00
25%,NaN,57.00,1.00
50%,NaN,66.00,2.00
75%,NaN,77.00,3.00


No missing values and no duplicate student records. The only real cleaning step is encoding
`parental level of education` as an ordered number (`edu_code`), since linear regression needs numeric
input. The six education levels have a natural order (`some high school` → ... → `master's degree`), so
an ordinal 0–5 code fits better here than one-hot encoding.

## Quick Look at the Data

In [5]:
edu_order = ["some high school", "high school", "some college",
             "associate's degree", "bachelor's degree", "master's degree"]

box_figure = px.box(df, x="parental level of education", y="math score",
                     category_orders={"parental level of education": edu_order},
                     title="Math Score Distribution by Parental Education", template="plotly_white",
                     color_discrete_sequence=["#2563eb"])
box_figure.update_layout(width=750, height=480, showlegend=False)
box_figure.show()

Median math score climbs step by step as parental education rises — roughly 63 for
`some high school` up to about 70 for `master's degree` — a mild but real upward trend, and the
reason an ordinal `edu_code` (rather than an unordered encoding) makes sense for this feature.

## B. Build the Linear Regression Model

In [6]:
X = df[["edu_code"]]
y = df["math score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"coefficient (edu_code): {model.coef_[0]:.2f}")
print(f"intercept (predicted score at edu_code=0, i.e. 'some high school'): {model.intercept_:.2f}")

edu_order = ["some high school", "high school", "some college",
             "associate's degree", "bachelor's degree", "master's degree"]
for code, label in enumerate(edu_order):
    predicted = model.intercept_ + model.coef_[0] * code
    print(f"edu_code={code} ({label}): predicted score = {predicted:.2f}")

coefficient (edu_code): 1.45
intercept (predicted score at edu_code=0, i.e. 'some high school'): 63.45
edu_code=0 (some high school): predicted score = 63.45
edu_code=1 (high school): predicted score = 64.90
edu_code=2 (some college): predicted score = 66.35
edu_code=3 (associate's degree): predicted score = 67.81
edu_code=4 (bachelor's degree): predicted score = 69.26
edu_code=5 (master's degree): predicted score = 70.71


With `edu_code` running 0–5, the fitted line now produces **six** possible predictions — one per
education level — each one step apart by the coefficient above. The positive coefficient means
predicted score rises with each step up in parental education, matching the boxplot trend.

## C. Measure the Model's Accuracy

In [7]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R^2:  {r2:.3f}   (fraction of variance in math score explained by parental education)")
print(f"MAE:  {mae:.2f}  points   (average absolute prediction error)")
print(f"RMSE: {rmse:.2f}  points   (typical prediction error, penalizes big misses more)")

R^2:  0.028   (fraction of variance in math score explained by parental education)
MAE:  12.14  points   (average absolute prediction error)
RMSE: 15.38  points   (typical prediction error, penalizes big misses more)


In [8]:
compare_figure = go.Figure()
compare_figure.add_trace(go.Scatter(
    x=df.loc[X_test.index, "parental level of education"], y=y_test, mode="markers",
    marker=dict(size=8, color="#94a3b8"), name="actual"))
compare_figure.add_trace(go.Scatter(
    x=edu_order, y=[model.intercept_ + model.coef_[0] * code for code in range(len(edu_order))],
    mode="markers+lines", marker=dict(size=14, color="#dc2626", symbol="x"),
    line=dict(color="#dc2626", width=2, dash="dash"), name="predicted"))
compare_figure.update_xaxes(categoryorder="array", categoryarray=edu_order)
compare_figure.update_layout(title="Actual Scores vs the Model's Predictions", xaxis_title="Parental level of education",
                              yaxis_title="Math score", template="plotly_white", width=750, height=480)
compare_figure.show()

## D. Summary

- **Data**: `parental level of education` and `math score` only, as scoped by the assignment. No missing
  values or duplicate rows in the raw data. `parental level of education` was encoded as an ordinal
  `edu_code` (0=`some high school` ... 5=`master's degree`) since linear regression needs numbers, and the
  six levels have a natural order.
- **Model**: simple linear regression with one ordinal feature, trained on an 80/20 split. Coefficient is
  positive — predicted score rises by roughly 1.45 points per step up the education ladder.
- **Accuracy**: R² ≈ 0.028 — parental education alone explains only a small slice of the variance in math
  scores; MAE ≈ 12 points and RMSE ≈ 15 points stay close to the overall score spread.
- **Takeaway**: a real but weak signal (correlation ≈ 0.16). Parental education nudges math score upward,
  but most of the variation comes from factors this single-feature model doesn't see (e.g. test preparation,
  individual study habits).